# Person 2 — Independent phenotype review and descriptive analysis

**Project:** MIMIC-IV-ED AKI prediction from triage information.

**Inputs:** Person 1's `aki_handoff/` export and extraction notebook
(`ML_for_health_Group_Project.ipynb`).

**Scope (from the team methodology document):**
1. Independently review the outcome code: 24-hour/discharge endpoint, strictly earlier
   comparators and exact time boundaries, decimal thresholds, conflicting samples,
   overlapping encounters and the operational-negative rule.
2. Check boundary cases with synthetic examples.
3. Characteristics table: primary positives vs operational negatives.
4. Labelled population vs indeterminate and no-usable-creatinine encounters (selection bias).
5. Outcome ascertainment summary.
6. First-sample groups among positives.
7. Candidate phenotype-sensitivity subsets (to be agreed with Person 5).

The unit of analysis is the ED encounter (`stay_id`). Every table states its encounter
denominator. Patient counts are given per group only; they must **not** be added across
groups because a patient can contribute encounters to several groups.

No predictors, splits or models are created here. Person 3 owns the split and preprocessing.

In [ ]:
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

# Works from this folder locally, or from /content on Colab
candidates = [Path("../aki_handoff"), Path("aki_handoff"), Path("/content/aki_handoff")]
DATA_DIR = next(p for p in candidates if (p / "modelling_data.csv").exists())

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

print("Data folder:", DATA_DIR.resolve())
print("pandas", pd.__version__)

## 1. Load the handoff files

CSV files do not keep pandas types. Two loading rules matter:

* Datetime fields in the audit table are parsed explicitly.
* **Raw text fields must be read with `keep_default_na=False`.** With pandas defaults,
  raw pain entries such as `"n/a"` and `"NA"` silently become missing values. That moves
  them from `other_recorded` to `missing` if anyone re-derives `pain_status` from the raw file.
  The cleaned `modelling_data.csv` is not affected, because `pain_status` is already stored.

In [ ]:
modelling = pd.read_csv(DATA_DIR / "modelling_data.csv")

time_cols = [
    "intime", "outtime", "admittime", "dischtime", "endpoint_time",
    "first_sample_time", "last_sample_time", "first_positive_time", "first_stage2_time",
]
audit = pd.read_csv(DATA_DIR / "full_cohort_audit.csv", low_memory=False, parse_dates=time_cols)

# Raw predictors: keep the original text exactly, only empty cells are missing
raw_labelled = pd.read_csv(
    DATA_DIR / "labelled_predictors_before_cleaning.csv",
    dtype=str, keep_default_na=False, na_values=[""],
)
raw_labelled["stay_id"] = raw_labelled["stay_id"].astype("int64")
raw_default = pd.read_csv(DATA_DIR / "labelled_predictors_before_cleaning.csv", dtype=str)

predictor_columns = pd.read_csv(DATA_DIR / "predictor_columns.csv")["predictor"].tolist()
outcome_counts = pd.read_csv(DATA_DIR / "outcome_counts.csv")
missingness_p1 = pd.read_csv(DATA_DIR / "predictor_missingness.csv", index_col=0)

print("modelling_data:", modelling.shape)
print("full_cohort_audit:", audit.shape)
print("labelled_predictors_before_cleaning:", raw_labelled.shape)
print("Predictors:", predictor_columns)

na_trap = raw_default["pain"].isna() & raw_labelled["pain"].notna()
print("\nRaw pain entries turned into NaN by default CSV parsing:", na_trap.sum())
print(raw_labelled.loc[na_trap, "pain"].value_counts().to_dict())

## 2. Count reconciliation

Every number in Person 1's README and in the methodology document is recomputed from the
exported files.

In [ ]:
def check(name, observed, expected):
    return {"check": name, "expected": expected, "observed": observed,
            "pass": bool(observed == expected)}

vitals = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp"]
labelled_audit = audit[audit["aki_label"].notna()]
positives_audit = audit[audit["label_state"] == "positive"]

recon = [
    check("Eligible encounters", len(audit), 195340),
    check("Eligible patients", audit["subject_id"].nunique(), 106516),
    check("Duplicate stay_id in audit", int(audit["stay_id"].duplicated().sum()), 0),
    check("Full 24-hour windows", int((audit["followup_hours"] == 24).sum()), 153501),
    check("Shorter windows (discharge < 24 h)", int((audit["followup_hours"] < 24).sum()), 41839),
    check("State: positive", int((audit["label_state"] == "positive").sum()), 7866),
    check("State: operational negative", int((audit["label_state"] == "negative").sum()), 105231),
    check("State: indeterminate", int((audit["label_state"] == "indeterminate").sum()), 72171),
    check("State: no usable current creatinine",
          int((audit["label_state"] == "no_usable_current_creatinine").sum()), 10072),
    check("outcome_counts.csv agrees with audit",
          outcome_counts.set_index("label_state")["encounters"].to_dict()
          == audit["label_state"].value_counts().to_dict(), True),
    check("Labelled encounters (modelling_data rows)", len(modelling), 113097),
    check("Labelled patients", modelling["subject_id"].nunique(), 66090),
    check("Duplicate stay_id in modelling_data", int(modelling["stay_id"].duplicated().sum()), 0),
    check("Primary positives (aki_label = 1)", int(modelling["aki_label"].sum()), 7866),
    check("Secondary positives (aki_stage2_label = 1)", int(modelling["aki_stage2_label"].sum()), 2254),
    check("Secondary positives outside primary positives",
          int(((modelling["aki_stage2_label"] == 1) & (modelling["aki_label"] == 0)).sum()), 0),
    check("Highest stage 1", int((positives_audit["highest_creatinine_stage"] == 1).sum()), 5612),
    check("Highest stage 2", int((positives_audit["highest_creatinine_stage"] == 2).sum()), 720),
    check("Highest stage 3", int((positives_audit["highest_creatinine_stage"] == 3).sum()), 1534),
    check("First sample already positive",
          int((positives_audit["first_sample_status"] == "positive").sum()), 2627),
    check("First sample assessable, no rise, later positive",
          int((positives_audit["first_sample_status"] == "no qualifying rise").sum()), 725),
    check("First sample unassessable, later positive",
          int((positives_audit["first_sample_status"] == "unassessable").sum()), 4514),
    check("Complete predictor records",
          int(modelling[predictor_columns].notna().all(axis=1).sum()), 89902),
    check("Encounters missing all six vital signs",
          int(modelling[vitals].isna().all(axis=1).sum()), 8823),
    check("Same stay_id set in modelling_data and labelled audit rows",
          set(modelling["stay_id"]) == set(labelled_audit["stay_id"]), True),
    check("Same stay_id order in modelling_data and raw predictors",
          bool((modelling["stay_id"].values == raw_labelled["stay_id"].values).all()), True),
    check("predictor_columns.csv has 13 fields", len(predictor_columns), 13),
    check("predictor_missingness.csv agrees with modelling_data",
          bool((missingness_p1["missing_count"] == modelling[predictor_columns].isna().sum()).all()), True),
]

# Labels in modelling_data must agree with the audit table
merged = modelling[["stay_id", "aki_label", "aki_stage2_label"]].merge(
    audit[["stay_id", "aki_label", "observed_stage2_positive"]],
    on="stay_id", suffixes=("", "_audit"), validate="one_to_one",
)
recon.append(check("aki_label disagreements with audit",
                   int((merged["aki_label"] != merged["aki_label_audit"]).sum()), 0))
recon.append(check("aki_stage2_label disagreements with audit",
                   int((merged["aki_stage2_label"] != merged["observed_stage2_positive"].astype(int)).sum()), 0))

recon = pd.DataFrame(recon)
recon.to_csv(OUT_DIR / "qa_count_reconciliation.csv", index=False, encoding="utf-8-sig")
print("All reconciliation checks pass:", recon["pass"].all())
recon

## 3. Rule checks on the audit table

The notebook derives each timing field and outcome state. Here they are recomputed
from the stored times and compared row by row. Each line counts violating encounters,
so 0 means the stored values agree with the stated rule.

In [ ]:
a = audit.copy()
hours = lambda td: td.dt.total_seconds() / 3600
tol = 1e-6
has_sample = a["current_sample_count"] > 0
pos = a["label_state"] == "positive"
neg = a["label_state"] == "negative"
ind = a["label_state"] == "indeterminate"
nouse = a["label_state"] == "no_usable_current_creatinine"
expected_endpoint = np.minimum(a["intime"] + pd.Timedelta(hours=24), a["dischtime"])

rules = {
    # Cohort eligibility
    "Age under 18": (a["age"] < 18),
    "Hospital discharge at or before ED arrival": (a["dischtime"] <= a["intime"]),
    "ED departure before arrival": (a["outtime"] < a["intime"]),
    "Prior ESKD or dialysis retained": (a["prior_eskd"] | a["prior_dialysis"]),
    # Observation window
    "endpoint_time != min(arrival + 24 h, discharge)": (a["endpoint_time"] != expected_endpoint),
    "followup_hours inconsistent with endpoint": ((hours(a["endpoint_time"] - a["intime"]) - a["followup_hours"]).abs() > tol),
    "followup_hours outside (0, 24]": ((a["followup_hours"] <= 0) | (a["followup_hours"] > 24)),
    # Sampling fields
    "Sample count > 0 but no first sample time": (has_sample & a["first_sample_time"].isna()),
    "Sample count = 0 but a first sample time": (~has_sample & a["first_sample_time"].notna()),
    "First sample before ED arrival": (a["first_sample_time"] < a["intime"]),
    "Last sample after endpoint": (a["last_sample_time"] > a["endpoint_time"]),
    "One sample but first != last time": ((a["current_sample_count"] == 1) & (a["first_sample_time"] != a["last_sample_time"])),
    "first_sample_delay_hours inconsistent": ((hours(a["first_sample_time"] - a["intime"]) - a["first_sample_delay_hours"]).abs() > tol),
    "sampling_span_hours inconsistent": ((hours(a["last_sample_time"] - a["first_sample_time"]) - a["sampling_span_hours"]).abs() > tol),
    "endpoint_gap_hours inconsistent": ((hours(a["endpoint_time"] - a["last_sample_time"]) - a["endpoint_gap_hours"]).abs() > tol),
    # Primary outcome
    "observed_positive disagrees with label_state": (a["observed_positive"] != pos),
    "Positive without a current sample": (pos & ~has_sample),
    "Positive time missing": (pos & a["first_positive_time"].isna()),
    "Positive time outside [arrival, endpoint]": (pos & ((a["first_positive_time"] < a["intime"]) | (a["first_positive_time"] > a["endpoint_time"]))),
    "Positive time outside [first, last] sample": (pos & ((a["first_positive_time"] < a["first_sample_time"]) | (a["first_positive_time"] > a["last_sample_time"]))),
    "hours_to_first_positive inconsistent": (pos & ((hours(a["first_positive_time"] - a["intime"]) - a["hours_to_first_positive"]).abs() > tol)),
    "First sample 'positive' but positive time != first sample time": ((a["first_sample_status"] == "positive") & (a["first_positive_time"] != a["first_sample_time"])),
    "Non-positive with a positive time": (~pos & a["first_positive_time"].notna()),
    # Operational negatives and unresolved states
    "Negative with < 2 current samples": (neg & (a["current_sample_count"] < 2)),
    "Negative without a timed comparison": (neg & ~a["has_timed_comparison"]),
    "Negative with an observed rise": (neg & a["observed_positive"]),
    "serial_no_rise disagrees with negative state": (a["serial_no_rise"] != neg),
    "Indeterminate with 0 samples": (ind & ~has_sample),
    "Indeterminate meeting the negative rule": (ind & (a["current_sample_count"] >= 2) & a["has_timed_comparison"]),
    "No-usable-creatinine state with samples": (nouse & has_sample),
    "first_sample_status 'no current sample' disagrees with count": ((a["first_sample_status"] == "no current sample") != ~has_sample),
    # Binary label
    "aki_label != 1 for positive": (pos & (a["aki_label"] != 1)),
    "aki_label != 0 for negative": (neg & (a["aki_label"] != 0)),
    "Unresolved encounter with a binary label": ((ind | nouse) & a["aki_label"].notna()),
    # Severity
    "Stage missing for positive": (pos & a["highest_creatinine_stage"].isna()),
    "Stage present for non-positive": (~pos & a["highest_creatinine_stage"].notna()),
    "Stage outside 1-3": (a["highest_creatinine_stage"].notna() & ~a["highest_creatinine_stage"].isin([1, 2, 3])),
    "Stage >= 2 flag disagrees with highest stage": (a["observed_stage2_positive"] != (a["highest_creatinine_stage"] >= 2).fillna(False)),
    "Stage >= 2 outside primary positives": (a["observed_stage2_positive"] & ~pos),
    "Stage >= 2 time before first positive time": (a["first_stage2_time"] < a["first_positive_time"]),
    "Stage >= 2 time after endpoint": (a["first_stage2_time"] > a["endpoint_time"]),
}

rule_checks = pd.DataFrame(
    [{"rule": k, "violating_encounters": int(v.fillna(False).sum())} for k, v in rules.items()]
)
rule_checks.to_csv(OUT_DIR / "qa_rule_checks.csv", index=False, encoding="utf-8-sig")
print("Encounters checked:", len(a))
print("All rule checks pass:", (rule_checks["violating_encounters"] == 0).all())
rule_checks

### Observations from the audit table (not errors)

In [ ]:
multi = a["current_sample_count"] >= 2
print("Encounters with >= 2 current samples:", multi.sum())
print("  ... of which lack a timed comparison:", (multi & ~a["has_timed_comparison"]).sum())
print("  -> With >= 2 distinct in-window times, the later sample always has the earlier one as a "
      "comparator. The 'usable timed comparison' condition is therefore always met and is redundant.\n")

single_pos = pos & (a["current_sample_count"] == 1)
single_norise = ind & (a["first_sample_status"] == "no qualifying rise")
print("Positives with only ONE current sample:", single_pos.sum(),
      f"({single_pos.sum() / pos.sum():.1%} of positives)")
print("One-sample encounters with an assessable comparison and NO rise "
      "(left indeterminate, not negative):", single_norise.sum())
print("  -> Positives can come from a single current sample compared with an earlier reference, "
      "but negatives need >= 2 current samples. This asymmetry is part of the agreed definition. "
      "It is quantified here and tested in sensitivity subset S4 below.")

## 4. Reproduce Person 1's predictor cleaning

The cleaning rules from notebook section 6 (README "Cleaning") are reimplemented
independently. They are then applied to `labelled_predictors_before_cleaning.csv` and the
result is compared with `modelling_data.csv`. The same function is reused in section 9 to
clean the excluded encounters for the selection-bias comparison.

In [ ]:
def clean_predictors(raw):
    """Apply the agreed section-6 cleaning rules to raw triage fields."""
    out = pd.DataFrame(index=raw.index)
    out["age"] = pd.to_numeric(raw["age"])
    out["gender"] = raw["gender"]
    out["arrival_transport"] = raw["arrival_transport"]

    temp = pd.to_numeric(raw["temperature"], errors="coerce")
    clean_temp = temp.where((temp > 70) & (temp < 120))
    celsius = temp.between(30, 43)
    clean_temp[celsius] = temp[celsius] * 9 / 5 + 32
    out["temperature"] = clean_temp

    upper = {"heartrate": 300, "resprate": 70, "sbp": 400, "dbp": 300}
    for col, limit in upper.items():
        v = pd.to_numeric(raw[col], errors="coerce")
        out[col] = v.where((v > 0) & (v < limit))
    o2 = pd.to_numeric(raw["o2sat"], errors="coerce")
    out["o2sat"] = o2.where((o2 > 0) & (o2 <= 100))

    acuity = pd.to_numeric(raw["acuity"], errors="coerce")
    out["acuity"] = acuity.where(acuity.between(1, 5))

    pain = raw["pain"].astype("string").str.strip()
    numeric = pain.str.fullmatch(r"([0-9]|10)(\.0+)?", na=False)
    out["pain_numeric"] = pd.to_numeric(pain.where(numeric), errors="coerce")
    missing = pain.isna() | pain.eq("").fillna(False)
    out["pain_status"] = "other_recorded"
    out.loc[missing, "pain_status"] = "missing"
    out.loc[numeric, "pain_status"] = "numeric"

    cc = raw["chiefcomplaint"].astype("string").str.strip()
    out["chiefcomplaint"] = cc.replace("", pd.NA)
    return out[[c for c in predictor_columns]]


recleaned = clean_predictors(raw_labelled)

cleaning_check = []
for col in predictor_columns:
    mine, theirs = recleaned[col], modelling[col]
    if pd.api.types.is_numeric_dtype(theirs):
        mine = mine.astype(float)
        same = (np.isclose(mine, theirs, rtol=0, atol=1e-9) | (mine.isna() & theirs.isna()))
    else:
        same = (mine.astype("string").fillna("<NA>") == theirs.astype("string").fillna("<NA>"))
    cleaning_check.append({"predictor": col, "rows_disagreeing": int((~same).sum())})
cleaning_check = pd.DataFrame(cleaning_check)
cleaning_check.to_csv(OUT_DIR / "qa_cleaning_reproduction.csv", index=False, encoding="utf-8-sig")

# Raw values in the audit table should equal the raw labelled export
raw_cols = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp", "acuity", "pain", "chiefcomplaint"]
audit_raw = pd.read_csv(DATA_DIR / "full_cohort_audit.csv", dtype=str, keep_default_na=False,
                        na_values=[""], usecols=["stay_id"] + raw_cols)
audit_raw["stay_id"] = audit_raw["stay_id"].astype("int64")
cmp = raw_labelled[["stay_id"] + raw_cols].merge(audit_raw, on="stay_id", suffixes=("", "_audit"))
audit_raw_mismatch = {c: int((cmp[c].fillna("<NA>") != cmp[c + "_audit"].fillna("<NA>")).sum()) for c in raw_cols}

print("Rows where independent cleaning disagrees with modelling_data:")
print(cleaning_check.set_index("predictor")["rows_disagreeing"].to_dict())
print("\nRaw values in audit vs raw labelled export (mismatching rows):", audit_raw_mismatch)

## 5. Synthetic boundary tests of the outcome logic

The source creatinine tables are available only inside the authorised BigQuery/Colab
environment. They are not in the handoff. So the outcome code is checked a different way.

`derive_outcomes()` below re-expresses notebook cells 27–64 (window → creatinine screen →
conflict removal → Decimal values → linkage → overlap assignment → comparators → rises →
stages → states) as one function. Synthetic patients are run through it. Each case targets
one boundary, and each expected result was worked out by hand from the written definition
in the methodology document.

In [ ]:
CURRENT_ITEMS = [50912, 52546]

def derive_outcomes(encounters, labs):
    """Re-expression of the notebook's outcome pipeline (cells 27-64)."""
    cohort = encounters.copy()

    # Observation window: earlier of 24 h after arrival and hospital discharge
    cohort["endpoint_time"] = cohort["intime"] + pd.Timedelta(hours=24)
    early = cohort["dischtime"] < cohort["endpoint_time"]
    cohort.loc[early, "endpoint_time"] = cohort.loc[early, "dischtime"]

    # Primary items, value screen, conflicting same-time values removed
    creat = labs[labs["itemid"].isin(CURRENT_ITEMS)].copy()
    v = pd.to_numeric(creat["valuenum"], errors="coerce")
    creat = creat[(v > 0) & (v <= 150)].copy()
    nvals = creat.groupby(["subject_id", "charttime"])["valuenum"].transform("nunique")
    creat = creat[nvals == 1].copy()
    creat["creat_decimal"] = creat["valuenum"].astype(str).map(Decimal)

    # Link samples to windows (matching or missing lab admission ID)
    windows = cohort[["subject_id", "stay_id", "hadm_id", "intime", "endpoint_time"]].rename(
        columns={"hadm_id": "ed_hadm_id"})
    m = creat[["labevent_id", "subject_id", "hadm_id", "charttime", "creat_decimal"]].merge(
        windows, on="subject_id")
    m = m[(m["charttime"] >= m["intime"]) & (m["charttime"] <= m["endpoint_time"])
          & (m["hadm_id"].isna() | (m["hadm_id"] == m["ed_hadm_id"]))]

    # One encounter per lab event: most recent arrival, then smaller stay_id
    ev = m.sort_values(["labevent_id", "intime", "stay_id"], ascending=[True, False, True]) \
          .drop_duplicates("labevent_id")
    cs = ev.drop_duplicates(["stay_id", "charttime"]).copy()
    cohort["current_sample_count"] = cohort["stay_id"].map(cs.groupby("stay_id").size()).fillna(0).astype(int)

    # Strictly earlier references within 48 h / 7 days (any of the patient's usable samples)
    ref = creat[["subject_id", "charttime", "creat_decimal"]].drop_duplicates(["subject_id", "charttime"]) \
        .rename(columns={"charttime": "reference_time", "creat_decimal": "reference_creatinine"})
    s = cs[["subject_id", "stay_id", "charttime", "creat_decimal"]].reset_index(drop=True)
    s["sample_id"] = s.index
    p = s.merge(ref, on="subject_id")
    gap = p["charttime"] - p["reference_time"]
    p = p[(gap > pd.Timedelta(0)) & (gap <= pd.Timedelta(days=7))]
    min7 = p.groupby("sample_id")["reference_creatinine"].min()
    min48 = p[(p["charttime"] - p["reference_time"]) <= pd.Timedelta(hours=48)] \
        .groupby("sample_id")["reference_creatinine"].min()
    s["minimum_48h"] = s["sample_id"].map(min48)
    s["minimum_7d"] = s["sample_id"].map(min7)

    # Qualifying rises (Decimal arithmetic)
    s["rise_48h"] = False
    s["rise_7d"] = False
    h48 = s["minimum_48h"].notna()
    s.loc[h48, "rise_48h"] = (s.loc[h48, "creat_decimal"] - s.loc[h48, "minimum_48h"]) >= Decimal("0.3")
    h7 = s["minimum_7d"].notna()
    s.loc[h7, "rise_7d"] = s.loc[h7, "creat_decimal"] >= s.loc[h7, "minimum_7d"] * Decimal("1.5")
    s["positive_sample"] = s["rise_48h"] | s["rise_7d"]
    s["has_comparator"] = h7

    # Stages
    s["stage"] = pd.Series(pd.NA, index=s.index, dtype="Int64")
    s.loc[s["positive_sample"], "stage"] = 1
    cur, refv = s.loc[h7, "creat_decimal"], s.loc[h7, "minimum_7d"]
    s.loc[cur[cur >= refv * Decimal("2")].index, "stage"] = 2
    s.loc[cur[cur >= refv * Decimal("3")].index, "stage"] = 3
    s.loc[s["positive_sample"] & (s["creat_decimal"] >= Decimal("4")), "stage"] = 3

    # Encounter-level results
    pos_stays = s.loc[s["positive_sample"], "stay_id"].unique()
    cohort["observed_positive"] = cohort["stay_id"].isin(pos_stays)
    cohort["highest_stage"] = cohort["stay_id"].map(s.groupby("stay_id")["stage"].max())
    cohort["stage2_label"] = (cohort["highest_stage"] >= 2).fillna(False).astype(int)

    first = s.sort_values("charttime").drop_duplicates("stay_id").copy()
    first["status"] = "unassessable"
    first.loc[first["has_comparator"] & ~first["positive_sample"], "status"] = "no qualifying rise"
    first.loc[first["positive_sample"], "status"] = "positive"
    cohort["first_sample_status"] = cohort["stay_id"].map(first.set_index("stay_id")["status"]) \
        .fillna("no current sample")

    has_cmp = cohort["stay_id"].map(s.groupby("stay_id")["has_comparator"].any()).eq(True)
    serial_no_rise = (cohort["current_sample_count"] >= 2) & has_cmp & ~cohort["observed_positive"]
    cohort["label_state"] = "indeterminate"
    cohort.loc[cohort["current_sample_count"] == 0, "label_state"] = "no_usable_current_creatinine"
    cohort.loc[serial_no_rise, "label_state"] = "negative"
    cohort.loc[cohort["observed_positive"], "label_state"] = "positive"
    return cohort

In [ ]:
T0 = pd.Timestamp("2150-01-01 08:00")
H = lambda h: pd.Timedelta(hours=h)
M = lambda mins: pd.Timedelta(minutes=mins)

encounters, labs, expected = [], [], []
lab_counter = [0]

def case(cid, description, stays, samples, expect):
    """stays: list of (stay_id, hadm_id, arrival offset, discharge offset).
    samples: list of (offset from T0, value, lab hadm_id or None, itemid).
    expect: {stay_id: (label_state, current_sample_count, highest_stage, first_sample_status)}."""
    subject = 1000 + cid
    for stay_id, hadm, arr, dis in stays:
        encounters.append({"case": cid, "subject_id": subject, "stay_id": stay_id, "hadm_id": hadm,
                           "intime": T0 + arr, "dischtime": T0 + dis})
    for offset, value, hadm, item in samples:
        lab_counter[0] += 1
        labs.append({"labevent_id": lab_counter[0], "subject_id": subject, "hadm_id": hadm,
                     "itemid": item, "charttime": T0 + offset, "valuenum": value})
    for stay_id, (state, n, stage, status) in expect.items():
        expected.append({"case": cid, "description": description, "stay_id": stay_id,
                         "exp_label_state": state, "exp_current_samples": n,
                         "exp_highest_stage": stage, "exp_first_sample_status": status})

CR = 50912
std = lambda sid: [(sid, sid, H(0), H(72))]  # one encounter; hadm_id = stay_id; discharge at 72 h

case(1, "Absolute rise of exactly 0.3 (2.0->2.3) that float arithmetic misses",
     std(101), [(H(1), 2.0, 101, CR), (H(5), 2.3, 101, CR)],
     {101: ("positive", 2, 1, "unassessable")})
case(2, "Rise of 0.29 (2.0->2.29) does not qualify",
     std(102), [(H(1), 2.0, 102, CR), (H(5), 2.29, 102, CR)],
     {102: ("negative", 2, None, "unassessable")})
case(3, "Reference exactly 48 h earlier (pre-arrival, lab hadm missing) is included",
     std(103), [(H(-46), 1.0, None, CR), (H(2), 1.3, 103, CR)],
     {103: ("positive", 1, 1, "positive")})
case(4, "Reference 48 h + 1 min earlier is excluded from the 48 h rule; ratio 1.3 < 1.5",
     std(104), [(H(-46) - M(1), 1.0, None, CR), (H(2), 1.3, 104, CR)],
     {104: ("indeterminate", 1, None, "no qualifying rise")})
case(5, "Reference exactly 7 days earlier with ratio exactly 1.5 qualifies",
     std(105), [(H(1) - pd.Timedelta(days=7), 1.0, None, CR), (H(1), 1.5, 105, CR)],
     {105: ("positive", 1, 1, "positive")})
case(6, "Reference 7 days + 1 min earlier is not a comparator",
     std(106), [(H(1) - pd.Timedelta(days=7) - M(1), 1.0, None, CR), (H(1), 1.5, 106, CR)],
     {106: ("indeterminate", 1, None, "unassessable")})
case(7, "Agreeing duplicate records at one time count once; same-time record is not a comparator",
     std(107), [(H(1), 1.0, 107, CR), (H(1), 1.0, 107, CR), (H(3), 1.0, 107, CR)],
     {107: ("negative", 2, None, "unassessable")})
case(8, "Conflicting values at one time are removed (as current samples and as references)",
     std(108), [(H(1), 1.0, 108, CR), (H(1), 2.0, 108, CR), (H(3), 1.0, 108, CR)],
     {108: ("indeterminate", 1, None, "unassessable")})
case(9, "Samples exactly at arrival and exactly at 24 h are both in the window",
     std(109), [(H(0), 1.0, 109, CR), (H(24), 1.5, 109, CR)],
     {109: ("positive", 2, 1, "unassessable")})
case(10, "Sample 1 min after 24 h is outside the window",
     std(110), [(H(0), 1.0, 110, CR), (H(24) + M(1), 1.5, 110, CR)],
     {110: ("indeterminate", 1, None, "unassessable")})
case(11, "Hospital discharge before 24 h shortens the window",
     [(111, 111, H(0), H(10))], [(H(1), 1.0, 111, CR), (H(12), 2.0, 111, CR)],
     {111: ("indeterminate", 1, None, "unassessable")})
case(12, "Lab from another admission is not a current sample but is a valid reference",
     std(112), [(H(2), 1.0, 999, CR), (H(6), 1.4, 112, CR)],
     {112: ("positive", 1, 1, "positive")})
case(13, "Overlapping encounters: shared sample goes to the most recent arrival",
     [(113, 1131, H(0), H(72)), (114, 1132, H(5), H(72))],
     [(H(2), 1.0, None, CR), (H(6), 1.4, None, CR), (H(8), 1.0, 1131, CR)],
     {113: ("negative", 2, None, "unassessable"), 114: ("positive", 1, 1, "positive")})
case(14, "Identical arrival times: shared samples go to the smaller stay_id",
     [(116, 1161, H(0), H(72)), (115, 1151, H(0), H(72))],
     [(H(1), 1.0, None, CR), (H(3), 1.5, None, CR)],
     {115: ("positive", 2, 1, "unassessable"), 116: ("no_usable_current_creatinine", 0, None, "no current sample")})
case(15, "Ratio exactly 2.0 gives stage 2",
     std(117), [(H(1), 1.0, 117, CR), (H(9), 2.0, 117, CR)],
     {117: ("positive", 2, 2, "unassessable")})
case(16, "Ratio exactly 3.0 gives stage 3",
     std(118), [(H(1), 1.0, 118, CR), (H(9), 3.0, 118, CR)],
     {118: ("positive", 2, 3, "unassessable")})
case(17, "Creatinine >= 4 with a qualifying 0.3 rise (3.7->4.0) gives stage 3",
     std(119), [(H(1), 3.7, 119, CR), (H(9), 4.0, 119, CR)],
     {119: ("positive", 2, 3, "unassessable")})
case(18, "Creatinine >= 4 without a qualifying rise (4.0->4.1) stays negative",
     std(120), [(H(1), 4.0, 120, CR), (H(9), 4.1, 120, CR)],
     {120: ("negative", 2, None, "unassessable")})
case(19, "Zero creatinine is screened out",
     std(121), [(H(1), 0.0, 121, CR), (H(3), 1.0, 121, CR)],
     {121: ("indeterminate", 1, None, "unassessable")})
case(20, "Whole-blood item 52024 is ignored for the primary endpoint",
     std(122), [(H(1), 1.0, 122, 52024), (H(3), 2.0, 122, CR)],
     {122: ("indeterminate", 1, None, "unassessable")})
case(21, "Item 52546 counts as a current sample",
     std(123), [(H(1), 1.0, 123, 52546), (H(3), 2.0, 123, CR)],
     {123: ("positive", 2, 2, "unassessable")})
case(22, "No creatinine in the window",
     std(124), [(H(-30), 1.0, None, CR)],
     {124: ("no_usable_current_creatinine", 0, None, "no current sample")})
case(23, "Already positive at first sample vs pre-arrival reference, later value falls",
     std(125), [(H(-24), 1.0, None, CR), (H(1), 1.6, 125, CR), (H(10), 1.2, 125, CR)],
     {125: ("positive", 2, 1, "positive")})
case(24, "Minimum (not most recent) reference is used",
     std(126), [(H(-30), 1.0, None, CR), (H(-2), 1.4, None, CR), (H(1), 1.35, 126, CR), (H(8), 1.36, 126, CR)],
     {126: ("positive", 2, 1, "positive")})

enc_df = pd.DataFrame(encounters)
lab_df = pd.DataFrame(labs)
lab_df["hadm_id"] = lab_df["hadm_id"].astype("Int64")
enc_df["hadm_id"] = enc_df["hadm_id"].astype("Int64")

result = derive_outcomes(enc_df.drop(columns="case"), lab_df)
tests = pd.DataFrame(expected).merge(
    result[["stay_id", "label_state", "current_sample_count", "highest_stage", "first_sample_status"]],
    on="stay_id")
tests["exp_highest_stage"] = tests["exp_highest_stage"].astype("Int64")
tests["pass"] = (
    (tests["label_state"] == tests["exp_label_state"])
    & (tests["current_sample_count"] == tests["exp_current_samples"])
    & ((tests["highest_stage"] == tests["exp_highest_stage"]).fillna(False)
       | (tests["highest_stage"].isna() & tests["exp_highest_stage"].isna()))
    & (tests["first_sample_status"] == tests["exp_first_sample_status"])
)
tests.to_csv(OUT_DIR / "qa_synthetic_boundary_tests.csv", index=False, encoding="utf-8-sig")
print(f"Synthetic cases passing: {tests['pass'].sum()} / {len(tests)}")
tests[["case", "description", "stay_id", "exp_label_state", "label_state",
       "exp_current_samples", "current_sample_count", "exp_highest_stage", "highest_stage", "pass"]]

**Why Decimal arithmetic matters.** Binary floating point gets these exact-threshold
cases wrong. Decimal gets them right:

In [ ]:
for cur, ref in [(2.3, 2.0), (4.0, 3.7), (0.7, 0.4)]:
    print(f"{cur} - {ref}: float -> {cur - ref!r} >= 0.3 is {cur - ref >= 0.3}; "
          f"Decimal -> {Decimal(str(cur)) - Decimal(str(ref)) >= Decimal('0.3')}")
for cur, ref in [(1.65, 1.1)]:
    print(f"{cur} vs 1.5 x {ref}: float -> {cur >= ref * 1.5}; "
          f"Decimal -> {Decimal(str(cur)) >= Decimal(str(ref)) * Decimal('1.5')}")

## 6. Code-review notes on the extraction notebook

Cells were read line by line against the methodology document.

| Area | Notebook cell(s) | Finding | Status |
|---|---|---|---|
| 24 h / discharge endpoint | 27, 40 | `min(intime + 24 h, dischtime)`. Samples are included when `intime <= charttime <= endpoint` (both ends inclusive). | Correct (tests 9–11) |
| Strictly earlier comparators | 49 | `0 < gap <= 7 d` and `gap <= 48 h`. Exact 48 h and 7 d references are included, and same-time references are excluded. | Correct (tests 3–7) |
| Decimal thresholds | 38, 51, 57 | Values are converted with `Decimal(str(valuenum))`. The 0.3, ×1.5, ×2, ×3 and ≥4 comparisons are exact. | Correct (tests 1, 5, 15–17) |
| Conflicting samples | 34–36 | Every record at a patient/time with more than one distinct value is dropped, so it is removed as a reference too. 960 times (1,926 records). | Matches document |
| Overlapping encounters | 42 | Most recent arrival wins, with smaller `stay_id` breaking ties. 456 lab events affected. | Correct (tests 13–14) |
| Admission matching | 40 | Lab `hadm_id` must equal the ED `hadm_id` or be missing, so outpatient/ED labs without `hadm_id` are kept. Labs from another admission are references only. | Correct (test 12) |
| Operational negative | 62–64 | `>= 2` distinct current times AND a timed comparison AND no rise. The comparison condition is always met when there are ≥ 2 distinct times, so it is redundant (0 exceptions). | Correct, redundant condition |
| Single-sample positives | 64 | 384 positives (4.9%) have one current sample and are labelled. 9,114 one-sample encounters with an assessable comparison and no rise are **indeterminate**. | Asymmetric ascertainment: documented, sensitivity S4 |
| Stage 3 absolute route | 57 | Requires creatinine ≥ 4.0 **and** a qualifying rise by either criterion. KDIGO's ≥ 4.0 route is usually paired with the absolute ≥ 0.3 rise, but the notebook also accepts a 7-day ×1.5 rise. Both are consistent with the written definition. | Acceptable |
| Prior ESKD/dialysis | 21–25 | ICD codes 5856, N186, V451, V4511, Z992 from a *different*, already-discharged admission. CKD stages 1–5 are retained. | Matches document |
| Age | 13 | `anchor_age + year(intime) − anchor_year`. This is an approximation, and anchor_age is top-coded in MIMIC. | Documented |
| Markdown text | 58 | Heading says "stage >2"; the code (correctly) uses ≥ 2. Several typos ("corhot", "asnd", "anad"). | Cosmetic |
| Notebook output | 40 | An encoding artefact appears in the printed label "Lab�Cencounter". | Cosmetic |
| CSV reload | export | Raw pain `"n/a"`/`"NA"` (24 rows) become NaN under default `read_csv`. Not an issue for `modelling_data.csv`. | Note for anyone re-cleaning from raw |
| Plausibility | 73 | Broad screens keep extreme values such as HR 1, SBP 1, SpO₂ 2 %, temperature 72 °F, and 59 rows with DBP ≥ SBP. | Agreed broad rule; Person 3 may wish to know |

**Overall:** no discrepancies were found that would change any count or label. All
reconciliation and rule checks pass, and the cleaning reproduces exactly.

## 7. Characteristics table: primary positives vs operational negatives

* Population: the labelled modelling cohort (`modelling_data.csv`, cleaned predictors).
* Denominators: encounters in each column.
* Continuous variables: median [IQR] among non-missing values, plus missing n (%).
* Categorical variables: n (%) of the column total.
* SMD: standardised mean difference, positives vs negatives. An absolute SMD above 0.1 is the
  usual threshold for a meaningful imbalance. SMDs are used instead of p-values because with
  more than 100,000 encounters almost every p-value is tiny.

The stage ≥2 column is a subset of the primary positives column.

In [ ]:
def fmt_n(n, d):
    return f"{n:,} ({100 * n / d:.1f}%)" if d else "–"

def med_iqr(x):
    x = x.dropna()
    if len(x) == 0:
        return "–"
    q1, q2, q3 = x.quantile([0.25, 0.5, 0.75])
    return f"{q2:.1f} [{q1:.1f}–{q3:.1f}]"

def smd_cont(x1, x0):
    x1, x0 = x1.dropna().astype(float), x0.dropna().astype(float)
    sd = np.sqrt((x1.var() + x0.var()) / 2)
    return (x1.mean() - x0.mean()) / sd if sd > 0 else np.nan

def smd_bin(b1, b0):
    p1, p0 = b1.mean(), b0.mean()
    sd = np.sqrt((p1 * (1 - p1) + p0 * (1 - p0)) / 2)
    return (p1 - p0) / sd if sd > 0 else np.nan

def describe_groups(df, groups, continuous, categorical, smd_pairs, labels=None):
    """groups: {column label: boolean mask}. smd_pairs: {smd label: (group a, reference group b)}."""
    labels = labels or {}
    rows = []
    header = {g: df[m] for g, m in groups.items()}
    rows.append({"variable": "Encounters, n", "level": "",
                 **{g: f"{len(d):,}" for g, d in header.items()},
                 **{k: "" for k in smd_pairs}})
    rows.append({"variable": "Patients, n (not additive across columns)", "level": "",
                 **{g: f"{d['subject_id'].nunique():,}" for g, d in header.items()},
                 **{k: "" for k in smd_pairs}})
    for var in continuous:
        name = labels.get(var, var)
        rows.append({"variable": name, "level": "median [IQR]",
                     **{g: med_iqr(d[var]) for g, d in header.items()},
                     **{k: f"{smd_cont(header[a][var], header[b][var]):.3f}" for k, (a, b) in smd_pairs.items()}})
        rows.append({"variable": name, "level": "missing",
                     **{g: fmt_n(int(d[var].isna().sum()), len(d)) for g, d in header.items()},
                     **{k: f"{smd_bin(header[a][var].isna(), header[b][var].isna()):.3f}" for k, (a, b) in smd_pairs.items()}})
    for var in categorical:
        name = labels.get(var, var)
        levels = df[var].astype("string").fillna("Missing").value_counts().index.tolist()
        for lev in levels:
            ind = {g: d[var].astype("string").fillna("Missing") == lev for g, d in header.items()}
            rows.append({"variable": name, "level": lev,
                         **{g: fmt_n(int(ind[g].sum()), len(header[g])) for g in header},
                         **{k: f"{smd_bin(ind[a], ind[b]):.3f}" for k, (a, b) in smd_pairs.items()}})
    return pd.DataFrame(rows)

labels = {
    "age": "Age, years", "gender": "Gender", "arrival_transport": "Arrival transport",
    "temperature": "Temperature, °F", "heartrate": "Heart rate, /min", "resprate": "Respiratory rate, /min",
    "o2sat": "Oxygen saturation, %", "sbp": "Systolic BP, mmHg", "dbp": "Diastolic BP, mmHg",
    "acuity_cat": "Triage acuity (ESI)", "acuity": "Triage acuity (ESI)", "pain_numeric": "Pain score (0–10)",
    "pain_status": "Pain entry", "all_vitals_missing": "All six vital signs missing",
    "race_group": "Race (descriptive only, not a predictor)", "anchor_year_group": "Anchor year group",
    "followup_hours": "Follow-up window, h", "ed_los_hours": "ED length of stay, h",
    "hosp_los_days": "Hospital length of stay, days", "discharged_lt24h": "Hospital discharge < 24 h after arrival",
}

md = modelling.copy()
md["acuity_cat"] = md["acuity"].map(lambda v: f"{int(v)}" if pd.notna(v) else pd.NA)
md["all_vitals_missing"] = np.where(md[vitals].isna().all(axis=1), "Yes", "No")

groups = {
    f"Operational negatives (N={int((md['aki_label'] == 0).sum()):,})": md["aki_label"] == 0,
    f"Primary positives (N={int((md['aki_label'] == 1).sum()):,})": md["aki_label"] == 1,
    f"  of which stage ≥2 (N={int((md['aki_stage2_label'] == 1).sum()):,})": md["aki_stage2_label"] == 1,
    f"All labelled (N={len(md):,})": pd.Series(True, index=md.index),
}
g = list(groups)
table1 = describe_groups(
    md, groups,
    continuous=["age"] + vitals + ["pain_numeric"],
    categorical=["gender", "arrival_transport", "acuity_cat", "pain_status", "all_vitals_missing"],
    smd_pairs={"SMD positive vs negative": (g[1], g[0])},
    labels=labels,
)
table1.to_csv(OUT_DIR / "table1_characteristics_pos_vs_neg.csv", index=False, encoding="utf-8-sig")
table1

### Small cells
Triage acuity 5 contains fewer than 10 encounters in some columns. Before publishing
aggregate tables, the team should decide whether to merge acuity 4–5 or suppress counts
below 10, in line with PhysioNet's MIMIC data use agreement. The check below lists every cell
with 1–9 encounters.

In [ ]:
def small_cells(table, cols):
    out = []
    for _, r in table.iterrows():
        for c in cols:
            s = str(r[c])
            if "(" in s:
                n = int(s.split(" ")[0].replace(",", ""))
                if 0 < n < 10:
                    out.append({"variable": r["variable"], "level": r["level"], "column": c.strip(), "n": n})
    return pd.DataFrame(out)

small_cells(table1, g)

## 8. Missingness table

Missingness of each model input after cleaning, by outcome. Missingness that differs by
outcome is informative. Person 3 should decide on missingness indicators with this in mind.

In [ ]:
miss_rows = []
neg_m, pos_m, s2_m = md["aki_label"] == 0, md["aki_label"] == 1, md["aki_stage2_label"] == 1
for col in predictor_columns + ["all six vital signs"]:
    if col == "all six vital signs":
        miss = md[vitals].isna().all(axis=1)
    else:
        miss = md[col].isna()
    miss_rows.append({
        "predictor": col,
        f"Negatives missing (N={neg_m.sum():,})": fmt_n(int(miss[neg_m].sum()), int(neg_m.sum())),
        f"Positives missing (N={pos_m.sum():,})": fmt_n(int(miss[pos_m].sum()), int(pos_m.sum())),
        f"Stage ≥2 missing (N={s2_m.sum():,})": fmt_n(int(miss[s2_m].sum()), int(s2_m.sum())),
        f"All labelled missing (N={len(md):,})": fmt_n(int(miss.sum()), len(md)),
        "SMD positive vs negative": round(smd_bin(miss[pos_m], miss[neg_m]), 3),
        "AKI rate if missing": f"{md.loc[miss, 'aki_label'].mean():.1%}" if miss.any() else "–",
        "AKI rate if observed": f"{md.loc[~miss, 'aki_label'].mean():.1%}",
    })
missingness = pd.DataFrame(miss_rows)
missingness.to_csv(OUT_DIR / "table2_missingness_by_outcome.csv", index=False, encoding="utf-8-sig")
print("Note: pain_numeric is missing for both 'missing' and 'other_recorded' pain entries; "
      "pain_status keeps them apart.")
missingness

## 9. Labelled population vs excluded encounters (selection)

Population: all 195,340 eligible encounters (`full_cohort_audit.csv`). Predictors in that
file are **before** cleaning, so the same `clean_predictors()` rules are applied first. This
matches the handling instruction in the methodology document.

SMDs compare each excluded group with the labelled population. Race, anchor-year group and
length of stay are shown for description only and are **not** model inputs. Length of stay
and follow-up time are measured after triage.

In [ ]:
audit_raw_full = pd.read_csv(
    DATA_DIR / "full_cohort_audit.csv", dtype=str, keep_default_na=False, na_values=[""],
    usecols=["stay_id", "age", "gender", "arrival_transport", "pain", "chiefcomplaint",
             "temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp", "acuity"],
)
audit_raw_full["stay_id"] = audit_raw_full["stay_id"].astype("int64")
cleaned_full = pd.concat([audit_raw_full[["stay_id"]], clean_predictors(audit_raw_full)], axis=1)

# The cleaned audit predictors must equal modelling_data for labelled encounters
chk = modelling[["stay_id"] + predictor_columns].merge(cleaned_full, on="stay_id", suffixes=("", "_re"))
num_cols = ["age", "temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp", "acuity", "pain_numeric"]
bad = sum(int((~(np.isclose(chk[c], chk[c + "_re"].astype(float), atol=1e-9)
                 | (chk[c].isna() & chk[c + "_re"].isna()))).sum()) for c in num_cols)
print("Cleaned audit predictors disagreeing with modelling_data (labelled rows):", bad)

def race_group(r):
    r = str(r).upper()
    if r.startswith("WHITE") or r == "PORTUGUESE": return "White"
    if r.startswith("BLACK"): return "Black"
    if r.startswith("HISPANIC") or r.startswith("SOUTH AMERICAN"): return "Hispanic/Latino"
    if r.startswith("ASIAN"): return "Asian"
    if r in ("UNKNOWN", "UNABLE TO OBTAIN", "PATIENT DECLINED TO ANSWER"): return "Unknown/declined"
    return "Other"

full = audit[["stay_id", "subject_id", "label_state", "race", "anchor_year_group", "followup_hours",
              "intime", "outtime", "admittime", "dischtime"]].merge(cleaned_full, on="stay_id", validate="one_to_one")
full["race_group"] = full["race"].map(race_group)
full["acuity_cat"] = full["acuity"].map(lambda v: f"{int(v)}" if pd.notna(v) else pd.NA)
full["all_vitals_missing"] = np.where(full[vitals].isna().all(axis=1), "Yes", "No")
full["discharged_lt24h"] = np.where(full["followup_hours"] < 24, "Yes", "No")
full["ed_los_hours"] = (full["outtime"] - full["intime"]).dt.total_seconds() / 3600
full["hosp_los_days"] = (full["dischtime"] - full["admittime"]).dt.total_seconds() / 86400

is_lab = full["label_state"].isin(["positive", "negative"])
is_ind = full["label_state"] == "indeterminate"
is_nou = full["label_state"] == "no_usable_current_creatinine"
sel_groups = {
    f"Labelled (N={is_lab.sum():,})": is_lab,
    f"Indeterminate (N={is_ind.sum():,})": is_ind,
    f"No usable creatinine (N={is_nou.sum():,})": is_nou,
    f"All eligible (N={len(full):,})": pd.Series(True, index=full.index),
}
sg = list(sel_groups)
table3 = describe_groups(
    full, sel_groups,
    continuous=["age"] + vitals + ["pain_numeric", "followup_hours", "ed_los_hours", "hosp_los_days"],
    categorical=["gender", "arrival_transport", "acuity_cat", "pain_status", "all_vitals_missing",
                 "discharged_lt24h", "race_group", "anchor_year_group"],
    smd_pairs={"SMD indeterminate vs labelled": (sg[1], sg[0]),
               "SMD no-creatinine vs labelled": (sg[2], sg[0])},
    labels=labels,
)
table3.to_csv(OUT_DIR / "table3_labelled_vs_excluded.csv", index=False, encoding="utf-8-sig")
table3

In [ ]:
# Variables with the largest imbalance (|SMD| > 0.1) to summarise selection
smd_long = table3.melt(id_vars=["variable", "level"],
                       value_vars=["SMD indeterminate vs labelled", "SMD no-creatinine vs labelled"],
                       var_name="comparison", value_name="smd")
smd_long = smd_long[smd_long["smd"] != ""].assign(smd=lambda d: d["smd"].astype(float))
big_imbalance = (smd_long[smd_long["smd"].abs() > 0.1]
                 .assign(abs_smd=lambda d: d["smd"].abs())
                 .sort_values(["comparison", "abs_smd"], ascending=[True, False])
                 .drop(columns="abs_smd"))
big_imbalance.to_csv(OUT_DIR / "table3b_selection_imbalances_smd_gt_0.1.csv", index=False, encoding="utf-8-sig")
big_imbalance

## 10. Outcome ascertainment summary

Population: all eligible encounters, by outcome state. Sampling-time fields are defined
only where samples exist (span and gap need ≥ 1 sample; span is 0 when there is one sample).

In [ ]:
def dist_row(name, series_by_group, fn):
    return {"measure": name, **{gname: fn(s) for gname, s in series_by_group.items()}}

state_order = ["positive", "negative", "indeterminate", "no_usable_current_creatinine"]
state_names = {"positive": "Primary positive", "negative": "Operational negative",
               "indeterminate": "Indeterminate", "no_usable_current_creatinine": "No usable creatinine"}
by_state = {f"{state_names[s]} (N={(audit['label_state'] == s).sum():,})": audit[audit["label_state"] == s]
            for s in state_order}

asc = []
asc.append(dist_row("Current samples, median [IQR]", {k: d["current_sample_count"] for k, d in by_state.items()}, med_iqr))
for lev, f in [("0", lambda x: x == 0), ("1", lambda x: x == 1), ("2", lambda x: x == 2),
               ("3", lambda x: x == 3), ("≥4", lambda x: x >= 4)]:
    asc.append(dist_row(f"Current samples = {lev}",
                        {k: d["current_sample_count"] for k, d in by_state.items()},
                        lambda s, f=f: fmt_n(int(f(s).sum()), len(s))))
asc.append(dist_row("First-sample delay from arrival, h", {k: d["first_sample_delay_hours"] for k, d in by_state.items()}, med_iqr))
asc.append(dist_row("Sampling span (first to last sample), h", {k: d["sampling_span_hours"] for k, d in by_state.items()}, med_iqr))
for lab, lo, hi in [("Sampling span < 6 h", 0, 6), ("Sampling span 6–<12 h", 6, 12), ("Sampling span ≥ 12 h", 12, np.inf)]:
    asc.append(dist_row(f"{lab} (among ≥2 samples)",
                        {k: d.loc[d["current_sample_count"] >= 2, "sampling_span_hours"] for k, d in by_state.items()},
                        lambda s, lo=lo, hi=hi: fmt_n(int(((s >= lo) & (s < hi)).sum()), len(s)) if len(s) else "–"))
asc.append(dist_row("Gap from last sample to endpoint, h", {k: d["endpoint_gap_hours"] for k, d in by_state.items()}, med_iqr))
asc.append(dist_row("Follow-up window, h", {k: d["followup_hours"] for k, d in by_state.items()}, med_iqr))
asc.append(dist_row("Full 24 h window", {k: d["followup_hours"] for k, d in by_state.items()},
                    lambda s: fmt_n(int((s == 24).sum()), len(s))))
for status in ["positive", "no qualifying rise", "unassessable", "no current sample"]:
    asc.append(dist_row(f"First sample status: {status}", {k: d["first_sample_status"] for k, d in by_state.items()},
                        lambda s, st=status: fmt_n(int((s == st).sum()), len(s))))
asc.append(dist_row("Hours from arrival to first qualifying sample", {k: d["hours_to_first_positive"] for k, d in by_state.items()}, med_iqr))
ascertainment = pd.DataFrame(asc)
ascertainment.to_csv(OUT_DIR / "table4_ascertainment_by_state.csv", index=False, encoding="utf-8-sig")
ascertainment

## 11. First-sample groups among primary positives

Denominator: 7,866 primary-positive encounters. Only the *already positive* group can be
AKI present at arrival. The first qualifying time is when the rise was ascertained, which is
not necessarily biological onset.

In [ ]:
p_ = audit[audit["label_state"] == "positive"].copy()
grp_names = {"positive": "Already positive at first sample",
             "no qualifying rise": "Assessable, no rise at first sample, later positive",
             "unassessable": "Unassessable at first sample, later positive"}
fs_rows = []
for key, name in grp_names.items():
    d = p_[p_["first_sample_status"] == key]
    fs_rows.append({
        "group": name,
        "encounters": len(d),
        "% of positives": f"{len(d) / len(p_):.1%}",
        "patients (not additive)": d["subject_id"].nunique(),
        "hours to first qualifying sample, median [IQR]": med_iqr(d["hours_to_first_positive"]),
        "first-sample delay h, median [IQR]": med_iqr(d["first_sample_delay_hours"]),
        "current samples, median [IQR]": med_iqr(d["current_sample_count"]),
        "single current sample": fmt_n(int((d["current_sample_count"] == 1).sum()), len(d)),
        "stage 1": fmt_n(int((d["highest_creatinine_stage"] == 1).sum()), len(d)),
        "stage 2": fmt_n(int((d["highest_creatinine_stage"] == 2).sum()), len(d)),
        "stage 3": fmt_n(int((d["highest_creatinine_stage"] == 3).sum()), len(d)),
        "stage ≥2": fmt_n(int(d["observed_stage2_positive"].sum()), len(d)),
    })
first_sample_groups = pd.DataFrame(fs_rows)
first_sample_groups.to_csv(OUT_DIR / "table5_first_sample_groups_positives.csv", index=False, encoding="utf-8-sig")
first_sample_groups.T

## 12. Candidate phenotype-sensitivity subsets (to agree with Person 5)

These are **proposals**. The methodology document asks for a small set, chosen before
anyone looks at test performance. Every subset is defined from extraction and ascertainment
fields only. Those fields must **never** be model inputs. They are exported as row-level flags
so that Persons 3–5 can apply a subset without recomputing anything. Patient-to-partition
assignment must not change.

| ID | Subset | Question it answers | Type |
|---|---|---|---|
| S1 | Exclude encounters missing all six vital signs | Does performance rely on encounters with almost no vital-sign information? | Evaluate frozen model in subgroup |
| S2 | Complete cases (all 13 predictors observed) | Effect of missing-data handling | Evaluate in subgroup (population changes) |
| S3 | Exclude positives already positive at the first current sample | Does the model mainly detect AKI present on arrival rather than AKI that develops later? | Evaluate in subgroup; retraining optional |
| S4 | Symmetric ascertainment: require ≥ 2 current samples for positives too | Removes the 384 single-sample positives that have no negative counterpart | Evaluate in subgroup |
| S5 | Stricter negatives: keep negatives only if sampling span ≥ 12 h; all positives kept | Sensitivity to under-ascertained negatives (short serial sampling) | Evaluate in subgroup. Note: prevalence rises |
| S6 | Full 24 h follow-up only (no discharge < 24 h) | Effect of truncated observation windows | Evaluate in subgroup |

Suggested minimum set: **S1, S3 and S4** (the doc's "missing vitals" and "stricter sampling"
items, plus the present-on-arrival question raised in the study aim).

In [ ]:
flags = modelling[["subject_id", "stay_id", "aki_label", "aki_stage2_label"]].merge(
    audit[["stay_id", "current_sample_count", "first_sample_status", "sampling_span_hours",
           "followup_hours"]], on="stay_id", validate="one_to_one")
flags["S1_has_any_vital"] = ~modelling[vitals].isna().all(axis=1).values
flags["S2_complete_case"] = modelling[predictor_columns].notna().all(axis=1).values
flags["S3_excl_positive_at_first_sample"] = ~((flags["aki_label"] == 1) & (flags["first_sample_status"] == "positive"))
flags["S4_two_or_more_samples"] = flags["current_sample_count"] >= 2
flags["S5_dense_negatives"] = (flags["aki_label"] == 1) | (flags["sampling_span_hours"] >= 12)
flags["S6_full_24h_followup"] = flags["followup_hours"] == 24

subset_cols = [c for c in flags.columns if c.startswith("S")]
sub_rows = [{"subset": "All labelled", "encounters": len(flags), "patients": flags["subject_id"].nunique(),
             "primary positives": int(flags["aki_label"].sum()),
             "primary prevalence": f"{flags['aki_label'].mean():.2%}",
             "stage ≥2 positives": int(flags["aki_stage2_label"].sum()),
             "stage ≥2 prevalence": f"{flags['aki_stage2_label'].mean():.2%}"}]
for c in subset_cols:
    d = flags[flags[c]]
    sub_rows.append({"subset": c, "encounters": len(d), "patients": d["subject_id"].nunique(),
                     "primary positives": int(d["aki_label"].sum()),
                     "primary prevalence": f"{d['aki_label'].mean():.2%}",
                     "stage ≥2 positives": int(d["aki_stage2_label"].sum()),
                     "stage ≥2 prevalence": f"{d['aki_stage2_label'].mean():.2%}"})
subset_counts = pd.DataFrame(sub_rows)
subset_counts.to_csv(OUT_DIR / "sensitivity_subset_counts.csv", index=False, encoding="utf-8-sig")

# Row-level flags: identifiers + booleans only (no predictors, no outcome times)
flags_out = flags[["subject_id", "stay_id"] + subset_cols]
flags_out.to_csv(OUT_DIR / "sensitivity_subset_flags.csv", index=False, encoding="utf-8-sig")
print("Row-level flags saved:", flags_out.shape, "- private storage only; NOT model inputs.")
subset_counts

## 13. One workbook with all aggregate tables (for report writing)

Row-level flags are left out on purpose.

In [ ]:
with pd.ExcelWriter(OUT_DIR / "P2_tables.xlsx") as xw:
    for name, df in [("qa_reconciliation", recon), ("qa_rule_checks", rule_checks),
                     ("qa_cleaning", cleaning_check), ("qa_synthetic_tests", tests),
                     ("T1_pos_vs_neg", table1), ("T2_missingness", missingness),
                     ("T3_labelled_vs_excluded", table3), ("T3b_selection_smd", big_imbalance),
                     ("T4_ascertainment", ascertainment), ("T5_first_sample_groups", first_sample_groups),
                     ("S_subset_counts", subset_counts)]:
        df.to_excel(xw, sheet_name=name, index=False)
print("Saved", OUT_DIR / "P2_tables.xlsx")

## 14. Output manifest

In [ ]:
for f in sorted(OUT_DIR.iterdir()):
    print(f"{f.name:55s} {f.stat().st_size / 1024:8.1f} KB")